# Analyse Helios Simulations
This notebook walks through the process of analysing helios simulations which follows 3 main steps:

    (1) Data Preparation
        This stage will setup the project directory, setup expected schemas for dataframes (both dask and pandas), and ultimately read in the helios data and prepare the required per ray information into a .parquet output.
        It will also setup the reference dataset for voxels for each voxel_size in the project (i.e. unique voxel_ids etc.).
    
    (2) Voxel-Ray Intersection
        With valid rays saved per leg of the scan, in the previous step, the goal now is to check ray intersections in all voxels. This will record important information, such as the entry/exit/hit coordinates of the ray which will later be used to gather metrics.
        The main reason these metrics are not gathered yet, is that this stage will remain separate per leg. That way, the metrics can be computed from different combinations of helios legs without re-computing voxel-ray intersections.

    (3) Compute Metrics
        Taking a given set of legs and voxel_sizes, the voxel_ray intersection files will be used to calculate metrics for each voxel, in this case resulting in all outputs from each investigated method.

## Step 1 - Data Preparation
This step focuses on converting helios simulation outputs, saving only valid rays into a more efficient .parquet file format.

It expects the following input and will add a new folder (valid_rays) to store all resulting .parquet files.

INPUT:
    project_dir/
    ├── reference/
    │   ├── "{project}_voxel_size_0.2.csv"
    │   ├── "{project}_voxel_size_0.5.csv"
    │   ...
    │   └── "{project}_voxel_size_{v}.csv"
    ├── helios/
    │   ├── "leg000_points.xyz"
    │   ├── "leg000_pulse.txt"
    │   ├── "leg000_fullwave.txt"
    │   ├── "leg001_points.xyz"
    │   ├── "leg001_pulse.txt"
    │   ├── "leg001_fullwave.txt"
    │   ├── ...
    │   ├── "leg{l}_points.xyz"
    │   ├── "leg{l}_pulse.txt"
    │   └── "leg{l}_fullwave.txt"

OUTPUT:
    └── valid_rays/
        ├── "leg_000_valid_rays.parquet"
        ├── "leg_001_valid_rays.parquet"
        ...
        └── "leg_{l}_valid_rays.parquet"

In [ ]:
import os
import utils

# Set up the project directory
project_dir = '/home/capheus/projects/helios_density_test/200_leaf_60/'
helios_dir = os.path.join(project_dir, 'helios')
references_dir = os.path.join(project_dir, 'references')

if not os.path.exists(helios_dir) or not os.path.exists(references_dir):
    raise FileNotFoundError("The specified directories do not exist. Please check the paths.")

valid_rays_dir = os.path.join(project_dir, 'valid_rays')
if not os.path.exists(valid_rays_dir):
    os.makedirs(valid_rays_dir)


# Establish the object IDs for the leaves
leaf_object_ids = [0]

# Run the data preparation script
utils.prepare_helios_data(input_dir=helios_dir, output_dir=valid_rays_dir, references_dir=references_dir, leaf_object_ids=leaf_object_ids)

## Step 2 - Voxel Ray Intersections
This code uses the valid rays from before, alongside the reference datasets in order to create a supporting parquet in the valid rays folder using the voxel_size_{voxel_size}_leg_{leg}_intersections.parquet format.

In [1]:
import os
import utils

# Set up the project directory
project_dir = '/home/capheus/projects/helios_density_test/200_leaf_60/'
valid_rays_dir = os.path.join(project_dir, 'valid_rays')
references_dir = os.path.join(project_dir, 'references')

# Run intersections
utils.voxel_ray_intersections(valid_rays_dir=valid_rays_dir, references_dir=references_dir) # Optional debug=False to disable logging

Processing chunk 1 of 2 for leg 4 and voxel size 0.2
[########################################] | 100% Completed | 31.37 s
Processing chunk 2 of 2 for leg 4 and voxel size 0.2
[########################################] | 100% Completed | 27.53 s
Processing chunk 1 of 1 for leg 4 and voxel size 0.5
[########################################] | 100% Completed | 10.07 s
Processing chunk 1 of 1 for leg 4 and voxel size 1.0
[########################################] | 100% Completed | 3.83 sms
Processing chunk 1 of 1 for leg 4 and voxel size 1.5
[########################################] | 100% Completed | 950.01 ms
Processing chunk 1 of 1 for leg 4 and voxel size 2.0
[########################################] | 100% Completed | 1.28 sms
Processing chunk 1 of 2 for leg 3 and voxel size 0.2
[########################################] | 100% Completed | 27.96 s
Processing chunk 2 of 2 for leg 3 and voxel size 0.2
[########################################] | 100% Completed | 27.07 s
Processing c

## Step 3 - Compute Metrics
Using the leg_{leg_id}_voxel_size_{voxel_size}_intersections.parquet files (which feature a standardised structure of columns from various inputs), compute the desired metrics and save outputs.

In [ ]:
import os
import glob
import utils
import pandas as pd

# Select the desired legs and voxel_sizes to include in the analysis
# Use the shortcut string 'all' to include all 
legs = 'all' # [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] 
voxel_sizes = 'all' # [0.1, 0.5, 1.0, 2.0]

# Set the average leaf area
average_leaf_area = 0.02  # in m^2, adjust as needed

# Set up the project directory
project_dir = '/home/capheus/projects/helios_density_test/200_leaf_60/'
valid_rays_dir = os.path.join(project_dir, 'valid_rays')
references_dir = os.path.join(project_dir, 'references')
# Set up the output directory
output_dir = os.path.join(project_dir, 'results')

# Create the output directory if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Get the list of all voxel sizes
intersection_files = []
if legs == 'all' and voxel_sizes == 'all':
    intersection_files = glob.glob(os.path.join(valid_rays_dir, '*_intersections.parquet'))
elif legs == 'all' and isinstance(voxel_sizes, list):
    for voxel_size in voxel_sizes:
        intersection_files += glob.glob(os.path.join(valid_rays_dir, f'leg_*_voxel_{voxel_size}_intersections.parquet'))
elif isinstance(legs, list) and voxel_sizes == 'all':
    for leg in legs:
        intersection_files += glob.glob(os.path.join(valid_rays_dir, f'leg_{leg}_*_intersections.parquet'))
else:
    for leg in legs:
        for voxel_size in voxel_sizes:
            intersection_files += glob.glob(os.path.join(valid_rays_dir, f'leg_{leg}_voxel_{voxel_size}_intersections.parquet'))

# Check if any intersection files were found
if intersection_files == []:
    print("No intersection files found. Please check the input parameters.")

# Split intersection files into separate lists for each voxel_size
voxel_size_files = {}
for file in intersection_files:
    # Extract the voxel size from the filename
    parts = file.split('_')
    voxel_size = float(parts[parts.index('voxel') + 1])
    
    # Add the file to the corresponding voxel size list
    if voxel_size not in voxel_size_files:
        voxel_size_files[voxel_size] = []
    voxel_size_files[voxel_size].append(file)

# Extract voxel information for each voxel size
for voxel_size, files in voxel_size_files.items():
    # Create a list of all legs in files
    legs = []
    for file in files:
        leg = os.path.basename(file)
        parts = leg.split('_')
        leg = int(parts[parts.index('leg') + 1])
        legs.append(leg)

    # Calculate the lambda_1 for average leaf area
    lambda_1 = utils.calculate_lambda_1(voxel_size=voxel_size, average_leaf_area=average_leaf_area)
    print(f"Voxel size: {voxel_size}, Lambda_1: {lambda_1}")

    # Calculate per voxel information from all files
    voxel_metrics_df = utils.get_voxel_metrics(intersections_files=files, lambda_1=lambda_1)

    # Retrieve the reference file
    reference_file = glob.glob(os.path.join(references_dir, f'*voxel_size_{voxel_size}*'))[0]
    df_ref = pd.read_csv(reference_file)
    df_ref = df_ref[['voxel_id', 'G_leaf_computed', 'G_lw_computed', 'CI_leaf', 'CI_lw', 'alpha', 'LAD_ref', 'PAD_ref']]
    df_ref.rename(columns={'G_leaf_computed': 'G_leaf', 'G_lw_computed': 'G_lw'}, inplace=True)
    # Remove duplicates and calculate mean values
    # This assumes that the reference file has a 'voxel_id' column
    # and that you want to average the values for each voxel_id
    df_ref = df_ref.groupby('voxel_id').mean().reset_index()
    df_ref = df_ref.add_suffix('_ref')
    df_ref.rename(columns={'voxel_id_ref': 'voxel_id', 'LAD_ref_ref': 'LAD_ref', 'PAD_ref_ref': 'PAD_ref'}, inplace=True)

    # Merge to maintain voxel_id matching
    voxel_metrics_df = voxel_metrics_df.merge(df_ref, on='voxel_id', how='left')

    ### LAD calculations
    # For simplicity, extract required variables
    LAD_ref = voxel_metrics_df['LAD_ref'].values

    I_leaf = voxel_metrics_df['I_leaf'].values
    mean_path_length = voxel_metrics_df['mean_path_length'].values  
    G_leaf = voxel_metrics_df['G_leaf'].values
    G_leaf_ref = voxel_metrics_df['G_leaf_ref'].values
    mean_eff_path_length = voxel_metrics_df['mean_eff_path_length'].values
    var_eff_path_length = voxel_metrics_df['var_eff_path_length'].values
    num_rays = voxel_metrics_df['num_rays'].values
    mean_free_path_length = voxel_metrics_df['mean_free_path_length'].values
    num_hits = voxel_metrics_df['num_hits'].values
    num_leaf_hits = voxel_metrics_df['num_leaf_hits'].values
    sum_eff_free_path_length = voxel_metrics_df['sum_eff_free_path_length'].values
    sum_eff_free_path_length_hit_leaf = voxel_metrics_df['sum_eff_free_path_length_hit_leaf'].values
    sum_free_path_length_hit = voxel_metrics_df['sum_free_path_length_hit'].values
    sum_free_path_length = voxel_metrics_df['sum_free_path_length'].values
    alpha_ref = voxel_metrics_df['alpha_ref'].values

    # LAD_BL
    LAD_BL_TLS = utils.BL_pimont_2018(I=I_leaf, mean_path_length=mean_path_length)
    LAD_BL_TLS_G = utils.BL_pimont_2018(I=I_leaf, mean_path_length=mean_path_length, G=G_leaf)
    LAD_BL_TLS_G_ref = utils.BL_pimont_2018(I=I_leaf, mean_path_length=mean_path_length, G=G_leaf_ref)
    # LAD_BL_TLS_CI_ref = utils.BL_pimont_2018(I=I_leaf, mean_path_length=mean_path_length, CI=CI_leaf_ref)

    voxel_metrics_df['LAD_BL_TLS'] = LAD_BL_TLS
    voxel_metrics_df['LAD_BL_TLS_G'] = LAD_BL_TLS_G
    voxel_metrics_df['LAD_BL_TLS_G_ref'] = LAD_BL_TLS_G_ref

    # LAD_EPL & UEPL
   
    LAD_EPL_TLS, LAD_UEPL_TLS = utils.BL_EPL_UEPL_pimont_2018(I=I_leaf, mean_eff_path_length=mean_eff_path_length, var_eff_path_length=var_eff_path_length, num_rays=num_rays)
    LAD_EPL_TLS_G, LAD_UEPL_TLS_G = utils.BL_EPL_UEPL_pimont_2018(I=I_leaf, mean_eff_path_length=mean_eff_path_length, var_eff_path_length=var_eff_path_length, num_rays=num_rays, G=G_leaf)
    LAD_EPL_TLS_G_ref, LAD_UEPL_TLS_G_ref = utils.BL_EPL_UEPL_pimont_2018(I=I_leaf, mean_eff_path_length=mean_eff_path_length, var_eff_path_length=var_eff_path_length, num_rays=num_rays, G=G_leaf_ref)

    voxel_metrics_df['LAD_EPL_TLS'] = LAD_EPL_TLS
    voxel_metrics_df['LAD_UEPL_TLS'] = LAD_UEPL_TLS
    voxel_metrics_df['LAD_EPL_TLS_G'] = LAD_EPL_TLS_G
    voxel_metrics_df['LAD_UEPL_TLS_G'] = LAD_UEPL_TLS_G
    voxel_metrics_df['LAD_EPL_TLS_G_ref'] = LAD_EPL_TLS_G_ref
    voxel_metrics_df['LAD_UEPL_TLS_G_ref'] = LAD_UEPL_TLS_G_ref

    # LAD_MCF
    
    LAD_MCF_TLS = utils.MCF_beland_2011(I=I_leaf, mean_free_path_length=mean_free_path_length)
    LAD_MCF_TLS_G = utils.MCF_beland_2011(I=I_leaf, mean_free_path_length=mean_free_path_length, G=G_leaf)
    LAD_MCF_TLS_G_ref = utils.MCF_beland_2011(I=I_leaf, mean_free_path_length=mean_free_path_length, G=G_leaf_ref)
    # LAD_MCF_TLS_CI_ref = utils.MCF_beland_2011(I=I_leaf, mean_path_length=mean_path_length, CI=CI_leaf_ref)

    voxel_metrics_df['LAD_MCF_TLS'] = LAD_MCF_TLS
    voxel_metrics_df['LAD_MCF_TLS_G'] = LAD_MCF_TLS_G
    voxel_metrics_df['LAD_MCF_TLS_G_ref'] = LAD_MCF_TLS_G_ref

    # LAD_MCF_Corr
    LAD_MCF_Corr_TLS = utils.MCF_corrected_beland_2014(I=I_leaf, mean_free_path_length=mean_free_path_length, lambda_1=lambda_1, mean_path_length=mean_path_length)
    LAD_MCF_Corr_TLS_G = utils.MCF_corrected_beland_2014(I=I_leaf, mean_free_path_length=mean_free_path_length, lambda_1=lambda_1, mean_path_length=mean_path_length, G=G_leaf)
    LAD_MCF_Corr_TLS_G_ref = utils.MCF_corrected_beland_2014(I=I_leaf, mean_free_path_length=mean_free_path_length, lambda_1=lambda_1, mean_path_length=mean_path_length, G=G_leaf_ref)

    voxel_metrics_df['LAD_MCF_Corr_TLS'] = LAD_MCF_Corr_TLS
    voxel_metrics_df['LAD_MCF_Corr_TLS_G'] = LAD_MCF_Corr_TLS_G
    voxel_metrics_df['LAD_MCF_Corr_TLS_G_ref'] = LAD_MCF_Corr_TLS_G_ref

    # LAD_MLE
    
    LAD_MLE_pimont_TLS = utils.MLE_pimont_2019(woody_vol_proportion=alpha_ref, num_hits=num_hits, num_leaf_hits=num_leaf_hits, sum_eff_free_path_length=sum_eff_free_path_length, sum_eff_free_path_length_hit_leaf=sum_eff_free_path_length_hit_leaf)
    LAD_MLE_pimont_TLS_G = utils.MLE_pimont_2019(woody_vol_proportion=alpha_ref, num_hits=num_hits, num_leaf_hits=num_leaf_hits, sum_eff_free_path_length=sum_eff_free_path_length, sum_eff_free_path_length_hit_leaf=sum_eff_free_path_length_hit_leaf, G=G_leaf)
    LAD_MLE_pimont_TLS_G_ref = utils.MLE_pimont_2019(woody_vol_proportion=alpha_ref, num_hits=num_hits, num_leaf_hits=num_leaf_hits, sum_eff_free_path_length=sum_eff_free_path_length, sum_eff_free_path_length_hit_leaf=sum_eff_free_path_length_hit_leaf, G=G_leaf_ref)

    voxel_metrics_df['LAD_MLE_pimont_TLS'] = LAD_MLE_pimont_TLS
    voxel_metrics_df['LAD_MLE_pimont_TLS_G'] = LAD_MLE_pimont_TLS_G
    voxel_metrics_df['LAD_MLE_pimont_TLS_G_ref'] = LAD_MLE_pimont_TLS_G_ref

    # LAD_MLE_Corr
    LAD_MLE_soma_TLS = utils.MLE_soma_2021(num_hits=num_hits, num_leaf_hits=num_leaf_hits, sum_free_path_length_hit=sum_free_path_length_hit, sum_free_path_length=sum_free_path_length)
    LAD_MLE_soma_TLS_G = utils.MLE_soma_2021(num_hits=num_hits, num_leaf_hits=num_leaf_hits, sum_free_path_length_hit=sum_free_path_length_hit, sum_free_path_length=sum_free_path_length, G=G_leaf)
    LAD_MLE_soma_TLS_G_ref = utils.MLE_soma_2021(num_hits=num_hits, num_leaf_hits=num_leaf_hits, sum_free_path_length_hit=sum_free_path_length_hit, sum_free_path_length=sum_free_path_length, G=G_leaf_ref)

    voxel_metrics_df['LAD_MLE_soma_TLS'] = LAD_MLE_soma_TLS
    voxel_metrics_df['LAD_MLE_soma_TLS_G'] = LAD_MLE_soma_TLS_G
    voxel_metrics_df['LAD_MLE_soma_TLS_G_ref'] = LAD_MLE_soma_TLS_G_ref

    # Save outputs to csv
    project_name = os.path.basename(os.path.normpath(project_dir))
    legs.sort()
    leg_string = "_".join(map(str, legs))
    output_file = os.path.join(output_dir, f"{project_name}_leg_{leg_string}_voxel_size_{voxel_size}.csv")
    if os.path.exists(output_file):
        os.remove(output_file)
    voxel_metrics_df.to_csv(output_file)


    






